# Sentiment Analysis

In [1]:
import os
print(os.path.abspath("arg_mining/ml_algorithms/ML/datasets/test.conll"))

c:\Users\huonc\Desktop\python\mining_project\arg_mining\ml_algorithms\ML\arg_mining\ml_algorithms\ML\datasets\test.conll


In [2]:
train_path = "datasets/train.conll"
test_path = "datasets/test.conll"
with open(train_path, encoding="utf-8") as f:
    train_data = f.read()

with open(test_path, encoding = "utf-8") as f:
    test_data = f.read()

In [3]:
#extract first column directly from conll files
def get_first_column(data):
    lines = data.strip().split('\n')
    first_col = []

    for i in lines:
        if i.strip():
            cols = i.split('\t')
            first_col.append(cols[0])

    return first_col

train_first_col = get_first_column(train_data)
test_first_col = get_first_column(test_data)

len(train_first_col), len(test_first_col)


(942743, 237236)

In [4]:
#join words to form sentence separated by tab
'''
open the conll file then remove whitespace, tabs, newline
there is only 1 column in the conll files so it'll just append those words into current_sentence
then join the current word to another word below to make a sentence
the final else indicates the last word of a sentence
'''
def read_conll_file(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            line = line.lower()
            line = line.strip()
            if line:
                parts = line.split('\t')
                word = parts[0]
                current_sentence.append(word)
            else:
                if current_sentence:
                    sentences.append(' '.join(current_sentence))
                    current_sentence = []

    if current_sentence:
        sentences.append(' '.join(current_sentence))
    return sentences

train_sentences = read_conll_file("datasets/train.conll")
test_sentences = read_conll_file("datasets/test.conll")
test_sentences[:10], train_sentences[:10]

(['comment wild times were living in romance and restroom breaks now go hand in hand',
  'now instead of trying to appease the crowds trump makes fun of their representative and major while also being incredibly authoritarian in his demand to respect federal authority and uniform',
  'comment i have never heard of that thats definitely not a thing here there were hundreds of people in my graduating class and i have no interest in any of them a group chat sounds like a nightmare',
  'why doesnt he wear cooler clothes',
  'anyone who is being picky about which democrat they will and wont vote for at this point isnt reading the room',
  '87 theriverismyhome',
  'comment gtit can be assumed that impeachment will not go through as dems do not have majority',
  '18 thankthebaker',
  '41 ghostofaugustwest',
  'and you just cant for instance pander towards religious muslims and lgbtq at the same time without that collapsing to its own internal contradictions at some point'],
 ['14 shelwood46',

In [5]:
#remove reddit usernames (elements start with a number)
train_sentences = [item for item in train_sentences if not item.split()[0].isdigit()]
test_sentences = [item for item in test_sentences if not item.split()[0].isdigit()]

train_sentences[:5], test_sentences[:5]

(['comment florida',
  'no you probably dont want a lawyer who has a facial tattoo saying bitch but i also dont care about someone having a couple of visible pieces even in professionals but thats just my own sensibility others have their own',
  'but indulge my thought process for a moment',
  'i was constantly losing my tie or forgetting to wear a belt with my pants and potentially getting in trouble for being out of uniform any idea that it eliminates dress code violations is a myth',
  'ive personally worked in jobs where anonymous client surveys can 100 lead to termination and those were just banking jobs the stress is genuine'],
 ['comment wild times were living in romance and restroom breaks now go hand in hand',
  'now instead of trying to appease the crowds trump makes fun of their representative and major while also being incredibly authoritarian in his demand to respect federal authority and uniform',
  'comment i have never heard of that thats definitely not a thing here th

In [6]:
#vocab to keep
negation_words = [
    "not", "no", "never", "none", "nothing", "neither", "nor",
    "hardly", "scarcely", "barely", "without"
]
intensifiers = [
    "very", "really", "extremely", "quite", "so", "too", "just",
    "absolutely", "totally", "incredibly", "barely", "fairly", "almost", "nearly"
]
modal_verbs = [
    "could", "would", "should", "might", "may", "must", "can", "shall", "will"
]
auxiliary_verbs = [
    "is", "are", "was", "were", "be", "been", "being", "am",
    "do", "does", "did", "have", "has", "had"
]
pronouns = [
    "i", "you", "we", "they", "he", "she", "it",
    "me", "us", "them", "my", "your", "our", "their",
    "mine", "yours", "his", "hers", "its"
]
conjunctions = [
    "but", "although", "though", "yet", "while", "whereas"
]
subjective_adverbs = [
    "always", "never", "sometimes", "often", "seldom",
    "unfortunately", "fortunately", "luckily", "sadly", "happily"
]
exception_words = (
    negation_words +
    intensifiers +
    modal_verbs +
    auxiliary_verbs +
    pronouns +
    conjunctions +
    subjective_adverbs
)
exception_words[:10]

['not',
 'no',
 'never',
 'none',
 'nothing',
 'neither',
 'nor',
 'hardly',
 'scarcely',
 'barely']

### Preprocess Text

1. source text
* data cleaning
    * identify noise
    * noise removal
    * character normalization
    * data masking*
* linguistic processing
    * tokenization
    * POS tagging
    * stopwords
    * lemmatization
    * named-entity recognition
        

In [7]:
import nltk 
#nltk.download('all')

In [8]:
import pandas as pd
import nltk 
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer


In [9]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    return " ".join(lemmatized_tokens)

train_processed = [preprocess_text(sentence) for sentence in train_sentences]
test_processed = [preprocess_text(sentence) for sentence in test_sentences]


In [10]:
len(train_processed), len(test_processed)

(29542, 7390)

In [11]:
#turn to pandas dataframe
df_train = pd.DataFrame({'sentences': train_processed})
df_test = pd.DataFrame({'sentences': test_processed})

In [12]:
analyzer = SentimentIntensityAnalyzer()
#
def get_sentiment(text):
    scores = analyzer.polarity_scores(text)
    if scores['pos'] > 0:
        sentiment = 1
    else:
        sentiment = 0
    return sentiment 

df_train['sentiment'] = df_train['sentences'].apply(get_sentiment)
df_test['sentiment'] = df_test['sentences'].apply(get_sentiment)

df_train.shape, df_test.shape
    

((29542, 2), (7390, 2))

In [13]:
from nltk.corpus import opinion_lexicon
negative_words = list(opinion_lexicon.negative())
negative_words[:10]

['2-faced',
 '2-faces',
 'abnormal',
 'abolish',
 'abominable',
 'abominably',
 'abominate',
 'abomination',
 'abort',
 'aborted']

In [14]:
#make a new column that shows positive sentence 
#expand contractions

#%pip install contractions
#%pip install swifter
import swifter
import contractions

negative_words = set(opinion_lexicon.negative())

def set_positive(sentence):
    expanded = contractions.fix(sentence)
    words = word_tokenize(expanded.lower())
    for i in words:
        if i in negative_words:
            return 0
    return 1

df_train['positive'] = df_train['sentences'].swifter.apply(set_positive)
df_test['positive'] = df_test['sentences'].swifter.apply(set_positive)
df_train.shape, df_test.shape
    

Pandas Apply:   0%|          | 0/29542 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/7390 [00:00<?, ?it/s]

((29542, 3), (7390, 3))

In [15]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(df_train['positive'], df_train['sentiment']))

[[ 3936 10302]
 [ 7893  7411]]


Confusion matrix: *many errors*

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_train['sentences'])
y_train = df_train['sentiment']

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

random_forest_classi = RandomForestClassifier(n_estimators=100, random_state = 42)
random_forest_classi.fit(X_train, y_train)

y_pred = random_forest_classi.predict(X_train)

conf_matrix = confusion_matrix(y_train, y_pred)
print("Confusion matrix: \n", conf_matrix)

print("\nClassification Report: \n", classification_report(y_train, y_pred))



Confusion matrix: 
 [[11829     0]
 [    0 17713]]

Classification Report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     11829
           1       1.00      1.00      1.00     17713

    accuracy                           1.00     29542
   macro avg       1.00      1.00      1.00     29542
weighted avg       1.00      1.00      1.00     29542



confusion matrix: *need to improve this unrealistic confusion matrix*